# 01 — Паспорт исходных данных

Этот ноутбук формирует только общую информацию о файлах для `docs/01_data.md`: пути, роли, размеры, схему и SHA-256. Он не анализирует распределения, пропуски или target.

Конфигурация файлов, key, target и описаний столбцов хранится централизованно в `src/ml_project/config.py`.

In [25]:
from pathlib import Path
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import (
    DataCatalog,
    DatasetProfiler,
    MarkdownDocument,
    build_data_blocks,
    build_eda_blocks,
    build_field_descriptions_template,
)
from ml_project.config import (
    DATASETS,
    FIELD_DESCRIPTIONS,
    INFERENCE_DATASET,
    KEY,
    RAW_DIR,
    TARGET,
    TRAIN_DATASET,
)

print(f"Корень проекта: {PROJECT_ROOT}")

Корень проекта: D:\Anton_ML\obsidian\Titanik-ML-project


In [26]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
datasets = catalog.load_all()

print("Загружены наборы:", ", ".join(datasets))

Загружены наборы: train, test, gender_submission


## 1. Реестр файлов

In [27]:
file_report = catalog.file_report()
display(
    file_report.drop(columns=["sha256"]).style.format(
        {"disk_kib": "{:.1f}", "memory_mib": "{:.3f}"}
    )
)

,dataset,file,role,rows,columns,disk_kib,memory_mib
0,train,data/raw/train.csv,train,891,12,59.8,0.308
1,test,data/raw/test.csv,inference,418,11,28.0,0.141
2,gender_submission,data/raw/gender_submission.csv,submission_example,418,2,3.2,0.007


## 2. Схема и роли столбцов

In [28]:
schema_report = catalog.schema_report(
    key=KEY,
    target=TARGET,
    inference_dataset=INFERENCE_DATASET,
    field_descriptions=FIELD_DESCRIPTIONS,
)
display(schema_report)

,dataset,field,dtype,description,role,available_at_inference
0,train,PassengerId,int64,уникальный id пассажира,id,yes
1,train,Survived,int64,"таргет 1-выжил, 0- не выжил",target,no
2,train,Pclass,int64,класс обслуживания,feature,yes
3,train,Name,str,Имя Пассажира,feature,yes
4,train,Sex,str,Пол,feature,yes
5,train,Age,float64,Возраст,feature,yes
6,train,SibSp,int64,"Наличие горизонтальной родни(брат, муж)",feature,yes
7,train,Parch,int64,"Наличие вертикальной родни(Дочь, Отец)",feature,yes
8,train,Ticket,str,Номер билета,feature,yes
9,train,Fare,float64,Стоимость проезда,feature,yes


## 3. Версия файлов

In [29]:
version_report = file_report[["dataset", "file", "sha256"]]
display(version_report)

,dataset,file,sha256
0,train,data/raw/train.csv,7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3...
1,test,data/raw/test.csv,56023b9948236f3c7a1c9448fcf418b283e109ef177fa8...
2,gender_submission,data/raw/gender_submission.csv,7a262a7f4cb807f53911dccc84257ed3976b21b7b50ffc...


## 4. Заготовка описаний полей

Ячейка автоматически находит все столбцы во входных файлах и печатает готовый словарь `FIELD_DESCRIPTIONS`. Уже заполненные описания сохраняются, новые поля получают пустую строку. `config.py` не изменяется автоматически: скопируйте заготовку только после проверки.


In [30]:
description_template = build_field_descriptions_template(
    catalog,
    FIELD_DESCRIPTIONS,
)
unconfigured_fields = list(
    dict.fromkeys(
        str(field)
        for dataset in datasets.values()
        for field in dataset.columns
        if not str(FIELD_DESCRIPTIONS.get(str(field), "")).strip()
    )
)

if unconfigured_fields:
    print(
        f"Без описания: {len(unconfigured_fields)}. "
        "Заполните пустые строки перед синхронизацией документа."
    )
else:
    print("Все найденные поля уже имеют описание.")

print("\nГотовая заготовка для src/ml_project/config.py:\n")
print(description_template)


Все найденные поля уже имеют описание.

Готовая заготовка для src/ml_project/config.py:

FIELD_DESCRIPTIONS = {
    'PassengerId': 'уникальный id пассажира ',
    'Survived': 'таргет 1-выжил, 0- не выжил',
    'Pclass': 'класс обслуживания',
    'Name': 'Имя Пассажира',
    'Sex': 'Пол',
    'Age': 'Возраст',
    'SibSp': 'Наличие горизонтальной родни(брат, муж)',
    'Parch': 'Наличие вертикальной родни(Дочь, Отец)',
    'Ticket': 'Номер билета',
    'Fare': 'Стоимость проезда',
    'Cabin': 'Номер каюты',
    'Embarked': 'Порт посадки \n C: (Шербур, Франция)\n Q:  (Квинстаун, Ирландия)\n S:(Саутгемптон, Англия)',
}


## 5. Применение конфига и синхронизация

После заполнения `src/ml_project/config.py` запустите **только следующую ячейку**. Она сама перечитает конфиг без перезапуска kernel, заново создаст каталог и обновит автоматические блоки в `docs/01_data.md`. Ручной текст документа сохраняется. Проблема в том, что импорты делаем вначале, и после коррекции, конфига, ничего в md автоматически не подставляется, поэтому нужно перечитать заново конфиг!

In [31]:
import importlib
import ml_project.config as project_config

# Перечитываем config.py, даже если модуль уже импортирован текущим kernel.
project_config = importlib.reload(project_config)

DATASETS = project_config.DATASETS
FIELD_DESCRIPTIONS = project_config.FIELD_DESCRIPTIONS
INFERENCE_DATASET = project_config.INFERENCE_DATASET
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET

# Пересоздаём все объекты, зависящие от конфигурации.
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
datasets = catalog.load_all()

unconfigured_fields = list(
    dict.fromkeys(
        str(field)
        for dataset in datasets.values()
        for field in dataset.columns
        if not str(FIELD_DESCRIPTIONS.get(str(field), "")).strip()
    )
)
if unconfigured_fields:
    print("Внимание: без описания —", ", ".join(unconfigured_fields))
if KEY is None:
    print("Внимание: KEY не настроен в config.py")
if TARGET is None:
    print("Внимание: TARGET не настроен в config.py")

data_blocks = build_data_blocks(
    catalog,
    key=KEY,
    target=TARGET,
    inference_dataset=INFERENCE_DATASET,
    field_descriptions=FIELD_DESCRIPTIONS,
)
updated_blocks = MarkdownDocument(
    PROJECT_ROOT / "docs" / "01_data.md"
).update_blocks(data_blocks)

print("Конфиг перечитан без перезапуска kernel.")
print("Загружены наборы:", ", ".join(datasets))
print("Обновлены блоки:", ", ".join(updated_blocks))

Конфиг перечитан без перезапуска kernel.
Загружены наборы: train, test, gender_submission
Обновлены блоки: data-file-report, data-schema, data-versions


## 6. Следующий шаг

После синхронизации проверьте `docs/01_data.md`, вручную заполните источник, правила доступа и ограничения. Когда Stage Gate выполнен, переходите к `notebooks/02_eda.ipynb`.